# QLoRA fine-tune of Qwen2.5-7B-Instruct

Runs on a free Colab T4. The notebook only orchestrates; the training code lives in
`src/` in the repository so it can be read and rerun outside Colab.

Runtime -> Change runtime type -> T4 GPU before starting.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

T4 is compute capability 7.5. That rules out bf16 and flash-attention, which is why
the training script runs fp16 with the sdpa attention backend.

In [ ]:
!pip install -q -U "transformers>=4.56,<6" "peft>=0.13" "bitsandbytes>=0.44" \
    "accelerate>=0.34" "datasets>=2.19" "wandb>=0.17"
import torch, transformers, peft, bitsandbytes
print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)

## Get the code

Fill in `REPO_URL` after pushing the repository. Until then, upload the project folder
to the Colab file browser and skip this cell.

In [ ]:
REPO_URL = ""   # e.g. https://github.com/<user>/<repo>.git

import os
if REPO_URL:
    !git clone -q $REPO_URL project
    %cd project
else:
    %cd /content/project
print(os.getcwd())
!ls src data

## Checkpoints on Drive

A free Colab session can be reclaimed mid-run. Writing checkpoints to Drive means a
disconnect costs one `--resume_from_checkpoint`, not the whole run. Skip if you would
rather not mount Drive.

In [ ]:
USE_DRIVE = True

CKPT_DIR = "adapter"
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = "/content/drive/MyDrive/chatdoctor-qlora"
    !mkdir -p "$CKPT_DIR"
print("checkpoints ->", CKPT_DIR)

## Weights & Biases

Paste the key from https://wandb.ai/authorize. The run URL it prints is the link the
task asks to submit.

In [ ]:
import wandb
wandb.login()

## Smoke run

Thirty examples, one epoch. Catches an OOM or a bad argument in a couple of minutes
instead of forty.

In [ ]:
!PYTHONPATH=src python src/train_lora.py \
    --limit-train 30 --limit-val 8 --epochs 1 --eval-steps 5 --save-steps 1000 \
    --out /tmp/smoke --no-wandb

## Full run

1000 examples, one epoch, effective batch 8 on a single T4, so about 125 optimiser
steps. The adapter kept is the best one by validation loss rather than the last: three
epochs overfits this data badly, see the README.

If the smoke run hit an out-of-memory error, add `--batch-size 1 --grad-accum 8`.

In [ ]:
!PYTHONPATH=src python src/train_lora.py \
    --out "$CKPT_DIR" \
    --epochs 1 --eval-steps 8 --load-best \
    --run-name qwen2.5-7b-chatdoctor-r16

## Loss curve

Saved locally as well as logged to W&B, so the repository carries the plot even if the
run link expires.

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

history = json.loads(Path(CKPT_DIR, "log_history.json").read_text())
train = [(h["step"], h["loss"]) for h in history if "loss" in h]
evals = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(*zip(*train), lw=1, alpha=.7, label="train")
if evals:
    ax.plot(*zip(*evals), marker="o", ms=4, label="validation")
ax.set_xlabel("step"); ax.set_ylabel("loss")
ax.set_title("Qwen2.5-7B-Instruct, QLoRA r=16")
ax.legend(); ax.grid(alpha=.3)
Path("reports").mkdir(exist_ok=True)
fig.tight_layout(); fig.savefig("reports/loss_curve.png", dpi=150)
print(f"final train {train[-1][1]:.4f}" + (f", final eval {evals[-1][1]:.4f}" if evals else ""))

## What was saved

Adapter only. The 15 GB of base weights stay on the Hub; this is the whole point of
LoRA and it is what the task asks to hand in.

In [ ]:
!du -sh "$CKPT_DIR"
!ls -la "$CKPT_DIR" 